# Mercy Fantasy Database

## Set Up

### Import

In [ ]:
import yahoo_fantasy_api as yfa
import nflreadpy as nfl
import pandas as pd
from yahoo_oauth import OAuth2
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, String, Float, Boolean, Date, BigInteger, DateTime
from datetime import datetime
import time

oauth = OAuth2(None, None, from_file="oauth2.json") # A file is needed to store authentication tokens for Yahoo connection
if not oauth.token_is_valid():
    oauth.refresh_access_token()
gm = yfa.Game(oauth, "nfl")

[2025-12-10 02:08:11,318 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2025-12-10 02:08:11,352 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 19383.668063879013
[2025-12-10 02:08:11,353 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN HAS EXPIRED
[2025-12-10 02:08:11,355 DEBUG] [yahoo_oauth.oauth.refresh_access_token] REFRESHING TOKEN
[2025-12-10 02:08:11,929 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 0.5737781524658203
[2025-12-10 02:08:11,931 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


### Database

In [141]:
# Database info
user = "mercyfantasy"
password = "Matthew23"
host = "localhost"
port = "5432"
database = "mercyfantasy"

# create a database connection
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{database}")

### Leagues

In [142]:
# Available Leagues
for year in (2023, 2024, 2025):
    league_ids = gm.league_ids(year)

    for lid in league_ids:
        lg = gm.to_league(lid)
        settings = lg.settings()  # includes league name + details
        print(f"League ID: {lid} | Name: {settings['name']} | Season: {settings['season']}")

League ID: 423.l.120539 | Name: New Years Weigh In | Season: 2023
League ID: 423.l.927065 | Name: DEM HICKSVILLE BOYS | Season: 2023
League ID: 449.l.91073 | Name: nick's Unbelievable League | Season: 2024
League ID: 449.l.978032 | Name: MERCY FANTASY | Season: 2024
League ID: 461.l.1135347 | Name: Mercy Fantasy | Season: 2025
League ID: 461.l.1388330 | Name: chris's Notable League | Season: 2025
League ID: 461.l.1390291 | Name: Montauk Monster | Season: 2025


In [143]:
# Relevant leagues
idMap = {2023: '423.l.927065', 2024: '449.l.978032', 2025: '461.l.1390291'}

## NFL Player Reference

In [162]:
# Load weekly player stats
weekly = nfl.load_player_stats(seasons=True)
weekly_df = weekly.to_pandas()

In [163]:
# Load weekly player stats
weekly = nfl.load_player_stats(seasons=True)
weekly_df = weekly.to_pandas()

# Filter dataset for 2023 onward and drop rows with missing position
weekly_df = weekly_df[(weekly_df['season'] >= 2023) & (weekly_df['position_group'].isin(['QB', 'RB', 'WR', 'TE', 'SPEC'])) & (weekly_df['position'] != 'P')].dropna(subset=['position']).reset_index(drop=True)
weekly_df = weekly_df[['season', 'week', 'team', 'opponent_team', 'player_display_name', 'position']]

# Fix LA to LAR for consistency
weekly_df[['team', 'opponent_team']] = (
    weekly_df[['team', 'opponent_team']]
        .replace({'LA': 'LAR'})
)

# Rename player_display_name to player_name
weekly_df = weekly_df.rename(columns={"player_display_name": "player_name"})

# Normalize names
weekly_df['player_name'] = weekly_df['player_name'].str.replace('.', '', regex=False)
weekly_df['player_name'] = weekly_df['player_name'].str.replace(r',?\s+(Jr|Sr|II|III|IV|V)$', '', regex=True)

# Fix specific name inconsistencies
name_corrections = {
    'Josh Palmer': 'Joshua Palmer',
    'Marquise Brown': 'Hollywood Brown',
    'Audric Estimé': 'Audric Estime'
}

weekly_df['player_name'] = weekly_df['player_name'].replace(name_corrections)

We did text cleaning here so we can relate this table to the roster table from yahoo. There are some slight discrepancies with suffixes and punctuation, so those will be stripped out. This relationship will allow us to get player team and opposing team for any player in a given week (which yahoo simply does not offer). For clarity, this is a record of players that played each week, but yahoo will have players with bye weeks, so those records will not have a join. That is desired behavior since when using this for modelling, a player on a bye week is not representative of their play at all. I also encoded some names that simply differ between the two tables to match yahoo. 

In [164]:
display(weekly_df.head())
print('Nulls:', weekly_df.isna().sum().sum())

,season,week,team,opponent_team,player_name,position
0,2023,1,NYJ,BUF,Aaron Rodgers,QB
1,2023,1,ARI,WAS,Matt Prater,K
2,2023,1,TEN,NO,Nick Folk,K
3,2023,1,LAR,SEA,Matthew Stafford,QB
4,2023,1,NYG,DAL,Graham Gano,K


Nulls: 0


In [ ]:
start = time.time()

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE roster_reference RESTART IDENTITY CASCADE;"))

weekly_df.to_sql(
    "roster_reference",
    engine,
    "player_info",
    if_exists="append",
    index=False,
    chunksize=5000,
    dtype={
    "season": Integer(),
    "week": Integer(),
    "team": String(10),
    "opponent_team": String(10),
    "player_name": String(50),
    "position": String(10)
    }
)

end = time.time()
print(f"Insertion completed in {end - start:.2f} seconds")

Insertion completed in 1.63 seconds


## Historical Data

### Matchups and Managers

In [ ]:
# Populate this array with each team's data for each matchup
rows = []
managers = []

# Iterate over each league and each week to get all matchups
for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    
    for week in range(1, end_week + 1):
        matchup = lg.matchups(week)["fantasy_content"]["league"][1]['scoreboard']
        for key, value in matchup['0']['matchups'].items():
            if key != 'count':
                match = value['matchup']
                if match['is_matchup_recap_available'] != 0:
                    week = match['week']
                    week_start = match['week_start']
                    week_end = match['week_end']
                    isPlayoff = match['is_playoffs']
                    teams = match['0']['teams']
                    
                    team1 = teams['0']['team'][0][2]['name']
                    team2 = teams['1']['team'][0][2]['name']

                    points1 = float(teams['0']['team'][1]['team_points']['total'])
                    points2 = float(teams['1']['team'][1]['team_points']['total'])

                    # Chat
                    recap_url = match['matchup_recap_url']
                    week = int(match['week'])
                    winner_key = match['winner_team_key']

                    # Each matchup has exactly two teams: '0' and '1'
                    for side in ['0', '1']:
                        team = teams[side]['team']
                        team_data = team[0]   # long nested metadata list
                        scoring = team[1]     # dict with points and projections

                        # --- Extract basic info ---
                        team_key = team_data[0]['team_key']
                        team_id = team_data[1]['team_id']
                        team_name = team_data[2]['name']
                        points = float(scoring['team_points']['total'])
                        proj_points = float(scoring['team_projected_points']['total'])

                        # --- Winner flag ---
                        winner = team_key == winner_key

                        # --- Extract manager info ---
                        manager_info = None
                        for item in team_data:
                            if isinstance(item, dict) and 'managers' in item:
                                manager_info = item['managers'][0]['manager']
                                break

                        manager_id = manager_info.get('manager_id') if manager_info else None
                        nickname = manager_info.get('nickname') if manager_info else None
                        guid = manager_info.get('guid') if manager_info else None

                        if week == 1:
                            num_moves, num_trades = None, None
                            for item in team_data:
                                if isinstance(item, dict):
                                    if 'number_of_moves' in item:
                                        num_moves = int(item['number_of_moves'])
                                    elif 'number_of_trades' in item:
                                        try:
                                            num_trades = int(item['number_of_trades'])
                                        except ValueError:
                                            num_trades = None
                            managers.append({
                                "manager_guid": guid,
                                "season": year,
                                "league_id": league_id,
                                "team_id": team_id,
                                "team_key": team_key,
                                "number_of_moves": num_moves,
                                "number_of_trades": num_trades,
                                'manager_nickname': nickname
                            })

                        # --- Add a flattened record ---
                        rows.append({
                            "season": year,
                            "week": week,
                            "week_start": week_start,
                            "week_end": week_end,
                            "is_playoff": isPlayoff,
                            "matchup_id": key,
                            "team_id": team_id,
                            "team_name": team_name,
                            "team_key": team_key,
                            "manager_id": manager_id,
                            "manager_nickname": nickname,
                            "manager_guid": guid,
                            "points": points,
                            "projected_points": proj_points,
                            "winner": winner,
                            "recap_url": recap_url
                        })

matchups_df = pd.DataFrame(rows)
matchups_df['game_quality'] = pd.cut(matchups_df['points'], bins =[0, 120, 150, float('inf')], labels = ['poor', 'average', 'strong'])
# I will be able to quickly filter for game quality level. I can easily tell who has the most of each, which is a helpful statistic for reporting.
manager_seasons_df = pd.DataFrame(managers)
managers_df = manager_seasons_df[['manager_guid', 'manager_nickname']].drop_duplicates().reset_index(drop=True)
manager_seasons_df = manager_seasons_df.drop(columns=['manager_nickname'])

In [20]:
display(managers_df)
print('Nulls:', managers_df.isna().sum().sum())

,manager_guid,manager_nickname
0,RLDAQUDDIH63SCRVGDUH7XHVVE,chris
1,GOBJY4A3RDGQYERM2ETTMRC7YY,Matthew
2,HL2OCHLAPFDTIFOI3ZENZETP5I,Skyler
3,2LZKPL56AY3GJ5QCXA6HWZOMGA,nick
4,Z7QJUKFA7ZPHDWIVNLXNVZZBVU,Billy
5,BB5X53X74V3GT4BZO4W4ZE3S2I,john
6,4EVL4NXFSEAMKJZTT2H5TBPXWI,David
7,ZFECT2SSXQHCUOGITGLWEYRHJE,Joseph
8,BHMUHFWNWP25IWXVETH4HHI7AY,Frankie
9,SZEJZ3NVHBN52EI2VCSK5Y4SQQ,daniel


Nulls: 0


In [21]:
display(matchups_df.tail(5))
print('Nulls:', matchups_df.isna().sum().sum())

,season,week,week_start,week_end,is_playoff,matchup_id,team_id,team_name,team_key,manager_id,manager_nickname,manager_guid,points,projected_points,winner,recap_url,game_quality
449,2025,13,2025-11-25,2025-12-01,0,2,8,Grit,461.l.1390291.t.8,8,Matthew,GOBJY4A3RDGQYERM2ETTMRC7YY,106.92,139.03,False,https://football.fantasysports.yahoo.com/f1/13...,poor
450,2025,13,2025-11-25,2025-12-01,0,3,5,Amon Ra Njigba,461.l.1390291.t.5,5,Billy,Z7QJUKFA7ZPHDWIVNLXNVZZBVU,120.92,153.99,False,https://football.fantasysports.yahoo.com/f1/13...,average
451,2025,13,2025-11-25,2025-12-01,0,3,9,Big 4🔒,461.l.1390291.t.9,9,Liam,BYOKM33VSC7PSCEB74CKX2E6FQ,124.84,144.08,True,https://football.fantasysports.yahoo.com/f1/13...,average
452,2025,13,2025-11-25,2025-12-01,0,4,6,Hail Mary john,461.l.1390291.t.6,6,john,BB5X53X74V3GT4BZO4W4ZE3S2I,110.64,132.91,False,https://football.fantasysports.yahoo.com/f1/13...,poor
453,2025,13,2025-11-25,2025-12-01,0,4,7,Wire Warriors,461.l.1390291.t.7,7,Joseph,ZFECT2SSXQHCUOGITGLWEYRHJE,130.00,137.27,True,https://football.fantasysports.yahoo.com/f1/13...,average


Nulls: 0


In [22]:
display(manager_seasons_df.head(5))
print('Nulls:', manager_seasons_df.isna().sum().sum())

,manager_guid,season,league_id,team_id,team_key,number_of_moves,number_of_trades
0,RLDAQUDDIH63SCRVGDUH7XHVVE,2023,423.l.927065,1,423.l.927065.t.1,65,10
1,GOBJY4A3RDGQYERM2ETTMRC7YY,2023,423.l.927065,5,423.l.927065.t.5,15,0
2,HL2OCHLAPFDTIFOI3ZENZETP5I,2023,423.l.927065,2,423.l.927065.t.2,33,3
3,2LZKPL56AY3GJ5QCXA6HWZOMGA,2023,423.l.927065,8,423.l.927065.t.8,21,5
4,Z7QJUKFA7ZPHDWIVNLXNVZZBVU,2023,423.l.927065,3,423.l.927065.t.3,12,5


Nulls: 0


In [ ]:
# Managers
start = time.time()
managers_df.to_sql('managers', engine, 'manager_info', if_exists='replace', index=False, dtype={
    "manager_guid": String(50),      # unique Yahoo GUID (e.g., 'GOBJY4A3RDGQYERM2ETTMRC7YY')
    "manager_nickname": String(50),  # manager's nickname ('Matthew')
    })

end = time.time()
print(f"Managers insertion completed in {end - start:.2f} seconds")

# Manager Seasons
start = time.time()
manager_seasons_df.to_sql('manager_seasons', engine, 'manager_info', if_exists='replace', index=False, dtype={
    "manager_guid": String(50),      # unique Yahoo GUID (e.g., 'GOBJY4A3RDGQYERM2ETTMRC7YY')
    "season": Integer(),             # Season year
    "league_id": String(50),         # numeric league ID
    "team_id": Integer(),            # numeric team ID
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "number_of_moves": Integer(),    # total moves made
    "number_of_trades": Integer()    # total trades made
    })
end = time.time()
print(f"Manager seasons insertion completed in {end - start:.2f} seconds")

# Matchups
start = time.time()
matchups_df.to_sql('matchups', engine, 'manager_info', if_exists='replace', index=False, chunksize=200, dtype={
    "season": Integer(),            # Season year
    "week": Integer(),              # week number (1–18)
    "week_start": Date(),           # start date of the week
    "week_end": Date(),             # end date of the week
    "is_playoff": Integer(),        # True if this week is in playoffs
    "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
    "team_id": Integer(),           # numeric team ID
    "team_name": String(100),       # e.g., "Hail Mary john"
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "manager_id": Integer(),        # numeric manager ID (if available)
    "manager_nickname": String(50), # e.g., "Matthew"
    "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
    "points": Float(),              # actual fantasy points scored
    "projected_points": Float(),    # projected points for the matchup
    "winner": String(10),           # 'win', 'loss', 'tie', or None
    "recap_url": String(255),       # URL to Yahoo recap (can be long)
    "game_quality": String(10)      # Designation of strong, average, or poor game
})
end = time.time()
print(f"Matchups insertion completed in {end - start:.2f} seconds")

454

These were populated before this project, so will not be re-populated now. Added timing code for assignment completeness.

### Transactions

In [23]:
parsed_trades = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    transactions = lg.transactions('trade', '1000')
    for trade in transactions:
        trade_id = trade['transaction_id']
        status = trade['status']
        timestamp = int(trade['timestamp'])
        players = trade['players']

        for k, player_entry in players.items():
            if k == 'count':
                continue

            # player info
            player_info = player_entry['player'][0]
            transaction_info = player_entry['player'][1]['transaction_data'][0]

            player_name = player_info[2]['name']['full']
            played_id = player_info[1]['player_id']
            position = player_info[4]['display_position']
            team_abbr = player_info[3]['editorial_team_abbr']

            from_team = transaction_info['source_team_name']
            from_team_key = transaction_info['source_team_key']
            to_team = transaction_info['destination_team_name']
            to_team_key = transaction_info['destination_team_key']

            parsed_trades.append({
                'season': year,
                'league_id': league_id,
                'trade_id': trade_id,
                'timestamp': timestamp,
                'status': status,
                'from_team': from_team,
                'from_team_key': from_team_key,
                'to_team': to_team,
                'to_team_key': to_team_key,
                'player': player_name,
                'player_id': played_id,
                'pos': position,
                'team_abbr': team_abbr
            })

trades = pd.DataFrame(parsed_trades)

# Convert timestamp to datetime
trades['timestamp'] = pd.to_datetime(trades['timestamp'], unit='s', utc=True)
trades['timestamp'] = trades['timestamp'].dt.tz_convert('America/New_York')
trades['player_id'] = trades['player_id'].astype(int)

In [26]:
display(trades.head(5))
print('Nulls:', trades.isna().sum().sum())

,season,league_id,trade_id,timestamp,status,from_team,from_team_key,to_team,to_team_key,player,player_id,pos,team_abbr
0,2023,423.l.927065,273,2023-11-16 12:10:31-05:00,successful,😈Griddy Goblin😈,423.l.927065.t.2,I HAVE 0 HOPE,423.l.927065.t.1,James Conner,30218,RB,Ari
1,2023,423.l.927065,273,2023-11-16 12:10:31-05:00,successful,😈Griddy Goblin😈,423.l.927065.t.2,I HAVE 0 HOPE,423.l.927065.t.1,Gabe Davis,32798,WR,Buf
2,2023,423.l.927065,273,2023-11-16 12:10:31-05:00,successful,I HAVE 0 HOPE,423.l.927065.t.1,😈Griddy Goblin😈,423.l.927065.t.2,Tony Pollard,31960,RB,Ten
3,2023,423.l.927065,265,2023-11-15 08:38:03-05:00,successful,Bust on Bijan,423.l.927065.t.10,john's Wondrous Team,423.l.927065.t.4,George Kittle,30259,TE,SF
4,2023,423.l.927065,265,2023-11-15 08:38:03-05:00,successful,Bust on Bijan,423.l.927065.t.10,john's Wondrous Team,423.l.927065.t.4,Najee Harris,33412,RB,LAC


Nulls: 0


In [ ]:
start = time.time()
trades.to_sql('trades', engine, 'transactions', if_exists='replace', index=False, chunksize=200, dtype={
    "season": Integer(),                    # year like 2025
    "league_id": String(50),                # league key (short ID)
    "trade_id": Integer(),                  # trade or transaction ID
    "timestamp": DateTime(timezone=True),   # UNIX timestamp of transaction
    "status": String(50),                   # e.g., 'successful'
    "from_team": String(100),               # name of team initiating transaction
    "from_team_key": String(100),           # key of team initiating transaction
    "to_team": String(100),                 # name of team receiving player
    "to_team_key": String(100),             # key of team receiving player
    "player": String(100),                  # player name
    "player_id": Integer(),                 # player ID
    "pos": String(10),                      # player position, e.g., 'DEF'
    "team_abbr": String(10)                 # team abbreviation, e.g., 'CHI'
})
end = time.time()
print(f"Insertion completed in {end - start:.2f} seconds")

Insertion completed in 0.26 seconds


In [27]:
parsed_waivers = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    transactions = lg.transactions('add', '5000')
    for trade in transactions:
        trade_id = trade['transaction_id']
        status = trade['status']
        timestamp = int(trade['timestamp'])
        players = trade['players']

        for k, player_entry in players.items():
            if k == 'count':
                continue

            # player info
            player_info = player_entry['player'][0]
            transaction_info = player_entry['player'][1]['transaction_data'][0] if isinstance(player_entry['player'][1]['transaction_data'], list) else player_entry['player'][1]['transaction_data']

            player_name = player_info[2]['name']['full']
            played_id = player_info[1]['player_id']
            position = player_info[4]['display_position']
            team_abbr = player_info[3]['editorial_team_abbr']
            type = transaction_info['type']
            add = type == 'add'
            teamType = 'destination' if add else 'source'

            team = transaction_info[f'{teamType}_team_name'] 
            team_key = transaction_info[f'{teamType}_team_key']

            parsed_waivers.append({
                'season': year,
                'league_id': league_id,
                'trade_id': trade_id,
                'timestamp': timestamp,
                'status': status,
                'type': type,
                'team': team,
                'team_key': team_key,
                'player': player_name,
                'player_id': played_id,
                'pos': position,
                'team_abbr': team_abbr
            })

waivers = pd.DataFrame(parsed_waivers)

# Convert timestamp to datetime and player_id to int
waivers['timestamp'] = pd.to_datetime(waivers['timestamp'], unit='s', utc=True)
waivers['timestamp'] = waivers['timestamp'].dt.tz_convert('America/New_York')
waivers['player_id'] = waivers['player_id'].astype(int)

In [28]:
display(waivers.head(5))
print('Nulls:', waivers.isna().sum().sum())

,season,league_id,trade_id,timestamp,status,type,team,team_key,player,player_id,pos,team_abbr
0,2023,423.l.927065,394,2023-12-30 18:06:38-05:00,successful,add,Frankie's Fantastic Team,423.l.927065.t.7,Derek Carr,27564,QB,NO
1,2023,423.l.927065,394,2023-12-30 18:06:38-05:00,successful,drop,Frankie's Fantastic Team,423.l.927065.t.7,Rashid Shaheed,34659,WR,Sea
2,2023,423.l.927065,393,2023-12-27 21:08:50-05:00,successful,add,Frankie's Fantastic Team,423.l.927065.t.7,Demarcus Robinson,29360,WR,SF
3,2023,423.l.927065,393,2023-12-27 21:08:50-05:00,successful,drop,Frankie's Fantastic Team,423.l.927065.t.7,Houston,100034,DEF,Hou
4,2023,423.l.927065,392,2023-12-27 21:07:45-05:00,successful,add,Frankie's Fantastic Team,423.l.927065.t.7,K.J. Osborn,32846,WR,Atl


Nulls: 0


In [ ]:
start = time.time()

waivers.to_sql('waivers', engine, 'transactions', if_exists='replace', index=False, chunksize=200, dtype={
    "season": Integer(),           # year like 2025
    "league_id": String(50),       # league key (short ID)
    "trade_id": String(50),        # trade or transaction ID
    "timestamp": DateTime(timezone=True),         # UNIX timestamp of transaction
    "status": String(50),          # e.g., 'successful'
    "type": String(50),            # transaction type, e.g., 'add/drop'
    "team": String(100),           # team name
    "team_key": String(100),       # team key
    "player": String(100),         # player name
    "player_id": Integer(),        # player ID
    "pos": String(10),             # player position, e.g., 'DEF'
    "team_abbr": String(10)        # team abbreviation, e.g., 'CHI'
})

end = time.time()
print(f"Insertion completed in {end - start:.2f} seconds")

Insertion completed in 0.24 seconds


### Detailed Roster

In [29]:
all_players = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    teams = lg.teams()
    for week in range(1, end_week + 1):
        for team_name, team in teams.items():
            roster = lg.to_team(team_name).roster(week=week)
            playerIds = [player["player_id"] for player in roster]
            stats = lg.player_stats(playerIds, req_type='week', week=week)
            for player, stat in zip(roster, stats):
                all_players.append({
                    "season": year,
                    "week": week,
                    "team_key": team_name,
                    "team_name": team['name'],
                    "manager_guid": team['managers'][0]['manager']['guid'],
                    "manager_name": team['managers'][0]['manager']['nickname'],
                    "player_id": player["player_id"],
                    "player_name": player["name"],
                    "played_position": player['selected_position'],
                    "position_type": stat['position_type'],
                    "pass_yds": stat['Pass Yds'] if 'Pass Yds' in stat else 0,
                    "pass_td": stat['Pass TD'] if 'Pass TD' in stat else 0,
                    "pass_int": stat['Int'] if 'Int' in stat and player['position_type'] == 'O' else 0,
                    "rush_att": stat['Rush Att'] if 'Rush Att' in stat else 0,
                    "rush_yds": stat['Rush Yds'] if 'Rush Yds' in stat else 0,
                    "rush_td": stat['Rush TD'] if 'Rush TD' in stat else 0,
                    "targets": stat['Targets'] if 'Targets' in stat else 0,
                    "rec": stat['Rec'] if 'Rec' in stat else 0,
                    "rec_yds": stat['Rec Yds'] if 'Rec Yds' in stat else 0,
                    "rec_td": stat['Rec TD'] if 'Rec TD' in stat else 0,
                    "offensive_ret_td": stat['Ret TD'] if 'Ret TD' in stat and player['position_type'] == 'O' else 0,
                    "two_pt_conversion": stat['2-PT'] if '2-PT' in stat else 0,
                    "fum_lost": stat['Fum Lost'] if 'Fum Lost' in stat else 0,
                    "fum_ret_td": stat['Fum Ret TD'] if 'Fum Ret TD' in stat else 0,
                    'fg_0_19': stat['FG 0-19'] if 'FG 0-19' in stat else 0,
                    'fg_20_29': stat['FG 20-29'] if 'FG 20-29' in stat else 0,
                    'fg_30_39': stat['FG 30-39'] if 'FG 30-39' in stat else 0,
                    'fg_40_49': stat['FG 40-49'] if 'FG 40-49' in stat else 0,
                    'fg_50_plus': stat['FG 50+'] if 'FG 50+' in stat else 0,
                    'pat_made': stat['PAT Made'] if 'PAT Made' in stat else 0,
                    'pts_allow': stat['Pts Allow'] if 'Pts Allow' in stat else 0,
                    'sack': stat['Sack'] if 'Sack' in stat else 0,
                    'dt_int': stat['Int'] if 'Int' in stat and player['position_type'] == 'DT' else 0,
                    'fum_rec': stat['Fum Rec'] if 'Fum Rec' in stat else 0,
                    'dt_td': stat['TD'] if 'TD' in stat and player['position_type'] == 'DT' else 0,
                    'safe': stat['Safe'] if 'Safe' in stat else 0,
                    'blk_kick': stat['Blk Kick'] if 'Blk Kick' in stat else 0,
                    'dt_ret_td': stat['Ret TD'] if 'Ret TD' in stat and player['position_type'] == 'DT' else 0,
                    'pts_allow_0': stat['Pts Allow 0'] if 'Pts Allow 0' in stat else 0,
                    'pts_allow_1_6': stat['Pts Allow 1-6'] if 'Pts Allow 1-6' in stat else 0,
                    'pts_allow_7_13': stat['Pts Allow 7-13'] if 'Pts Allow 7-13' in stat else 0,
                    'pts_allow_14_20': stat['Pts Allow 14-20'] if 'Pts Allow 14-20' in stat else 0,
                    'pts_allow_21_27': stat['Pts Allow 21-27'] if 'Pts Allow 21-27' in stat else 0,
                    'pts_allow_28_34': stat['Pts Allow 28-34'] if 'Pts Allow 28-34' in stat else 0,
                    'pts_allow_35_plus': stat['Pts Allow 35+'] if 'Pts Allow 35+' in stat else 0,
                    'xpr': stat['XPR'] if 'XPR' in stat else 0,
                    "points": stat['total_points'] if 'total_points' in stat else 0,
                })

detailed_roster = pd.DataFrame(all_players)

# Normalize names
detailed_roster['player_name'] = detailed_roster['player_name'].str.replace('.', '', regex=False)
detailed_roster['player_name'] = detailed_roster['player_name'].str.replace(r',?\s+(Jr|Sr|II|III|IV|V)$', '', regex=True)

In [30]:
display(detailed_roster.head(5))
print('Nulls:', detailed_roster.isna().sum().sum())

,season,week,team_key,team_name,manager_guid,manager_name,player_id,player_name,played_position,position_type,...,dt_ret_td,pts_allow_0,pts_allow_1_6,pts_allow_7_13,pts_allow_14_20,pts_allow_21_27,pts_allow_28_34,pts_allow_35_plus,xpr,points
0,2023,1,423.l.927065.t.7,Frankie's Fantastic Team,BHMUHFWNWP25IWXVETH4HHI7AY,Frankie,31002,Lamar Jackson,QB,O,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.56
1,2023,1,423.l.927065.t.7,Frankie's Fantastic Team,BHMUHFWNWP25IWXVETH4HHI7AY,Frankie,33966,Chris Olave,WR,O,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.20
2,2023,1,423.l.927065.t.7,Frankie's Fantastic Team,BHMUHFWNWP25IWXVETH4HHI7AY,Frankie,32703,Tee Higgins,WR,O,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
3,2023,1,423.l.927065.t.7,Frankie's Fantastic Team,BHMUHFWNWP25IWXVETH4HHI7AY,Frankie,33394,Jaylen Waddle,WR,O,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.80
4,2023,1,423.l.927065.t.7,Frankie's Fantastic Team,BHMUHFWNWP25IWXVETH4HHI7AY,Frankie,30121,Christian McCaffrey,RB,O,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.90


Nulls: 0


Normalized player names on yahoo side to match with normalized names from NFL reference data.

In [ ]:
start = time.time()

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE detailed_roster RESTART IDENTITY CASCADE;"))

detailed_roster.to_sql('detailed_roster', engine, 'player_info', if_exists='append', chunksize=1000, index=False,
    dtype={
        "season": Integer(),
        "week": Integer(),
        "team_key": String(100),
        "team_name": String(100),
        "manager_guid": String(50),
        "manager_name": String(50),
        "player_id": Integer(),
        "player_name": String(100),
        "played_position": String(10),
        "position_type": String(5),

        # Passing
        "pass_yds": Integer(),
        "pass_td": Integer(),
        "pass_int": Integer(),

        # Rushing
        "rush_att": Integer(),
        "rush_yds": Integer(),
        "rush_td": Integer(),

        # Receiving
        "targets": Integer(),
        "rec": Integer(),
        "rec_yds": Integer(),
        "rec_td": Integer(),

        # Misc offensive
        "offensive_ret_td": Integer(),
        "two_pt_conversion": Integer(),
        "fum_lost": Integer(),
        "fum_ret_td": Integer(),

        # Kicker
        "fg_0_19": Integer(),
        "fg_20_29": Integer(),
        "fg_30_39": Integer(),
        "fg_40_49": Integer(),
        "fg_50_plus": Integer(),
        "pat_made": Integer(),
        "xpr": Integer(),

        # Defense
        "pts_allow": Integer(),
        "sack": Integer(),
        "dt_int": Integer(),
        "fum_rec": Integer(),
        "dt_td": Integer(),
        "safe": Integer(),
        "blk_kick": Integer(),
        "dt_ret_td": Integer(),

        # Defense point tiers
        "pts_allow_0": Integer(),
        "pts_allow_1_6": Integer(),
        "pts_allow_7_13": Integer(),
        "pts_allow_14_20": Integer(),
        "pts_allow_21_27": Integer(),
        "pts_allow_28_34": Integer(),
        "pts_allow_35_plus": Integer(),

        # Final fantasy points
        "points": Float()
    }
)

end = time.time()
print(f"Insertion completed in {end - start:.2f} seconds")

Insertion completed in 3.92 seconds


## Weekly Appends

In [144]:
currentSeason = max(list(idMap.keys()))

In [145]:
lg = gm.to_league(idMap[currentSeason])
settings = lg.settings()
current_week = lg.current_week()

### Current Matchup

In [146]:
rows = []
matchup = lg.matchups(current_week)["fantasy_content"]["league"][1]['scoreboard']

for key, value in matchup['0']['matchups'].items():
    if key != 'count':
        match = value['matchup']
        week = match['week']
        week_start = match['week_start']
        week_end = match['week_end']
        isPlayoff = match['is_playoffs']
        teams = match['0']['teams']
        
        team1 = teams['0']['team'][0][2]['name']
        team2 = teams['1']['team'][0][2]['name']

        # Each matchup has exactly two teams: '0' and '1'
        for side in ['0', '1']:
            team = teams[side]['team']
            team_data = team[0]   # long nested metadata list
            scoring = team[1]     # dict with points and projections

            # --- Extract basic info ---
            team_key = team_data[0]['team_key']
            team_id = team_data[1]['team_id']
            team_name = team_data[2]['name']
            proj_points = float(scoring['team_projected_points']['total'])

            manager_info = None
            for item in team_data:
                if isinstance(item, dict) and 'managers' in item:
                    manager_info = item['managers'][0]['manager']
                    break

            manager_id = manager_info.get('manager_id') if manager_info else None
            nickname = manager_info.get('nickname') if manager_info else None
            guid = manager_info.get('guid') if manager_info else None

            # --- Add a flattened record ---
            rows.append({
                "season": currentSeason,
                "week": week,
                "week_start": week_start,
                "week_end": week_end,
                "is_playoff": isPlayoff,
                "matchup_id": key,
                "team_id": team_id,
                "team_name": team_name,
                "team_key": team_key,
                "manager_id": manager_id,
                "manager_nickname": nickname,
                "manager_guid": guid,
                "projected_points": proj_points,
            })

currentMatchup = pd.DataFrame(rows)

In [148]:
start = time.time()

# Use this truncate table command to replace the data in the currentmatchup table each week
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE manager_info.currentmatchup RESTART IDENTITY CASCADE;"))

currentMatchup.to_sql('currentmatchup', engine, 'manager_info', if_exists='append', index=False, dtype={
    "season": Integer(),            # Season
    "week": Integer(),              # week number (1–18)
    "week_start": Date(),           # start date of the week
    "week_end": Date(),             # end date of the week
    "is_playoff": Integer(),        # True if this week is in playoffs
    "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
    "team_id": Integer(),           # numeric team ID
    "team_name": String(100),       # e.g., "Hail Mary john"
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "manager_id": Integer(),        # numeric manager ID (if available)
    "manager_nickname": String(50), # e.g., "Matthew"
    "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
    "projected_points": Float()    # projected points for the matchup
})

end = time.time()
print(f"Insertion completed in {end - start:.2f} seconds")

Insertion completed in 0.29 seconds


### Append Last Weeks Data

In [149]:
previousWeek = current_week - 1 if current_week != 17 else 17

In [160]:
if previousWeek != 0:
    rows = []
    matchup = lg.matchups(previousWeek)["fantasy_content"]["league"][1]['scoreboard']

    for key, value in matchup['0']['matchups'].items():
        if key != 'count':
            match = value['matchup']
            if match['is_matchup_recap_available'] != 0:
                week = match['week']
                week_start = match['week_start']
                week_end = match['week_end']
                isPlayoff = match['is_playoffs']
                teams = match['0']['teams']
                
                team1 = teams['0']['team'][0][2]['name']
                team2 = teams['1']['team'][0][2]['name']

                points1 = float(teams['0']['team'][1]['team_points']['total'])
                points2 = float(teams['1']['team'][1]['team_points']['total'])

                # Chat
                recap_url = match['matchup_recap_url']
                week = int(match['week'])
                winner_key = match['winner_team_key']

                # Each matchup has exactly two teams: '0' and '1'
                for side in ['0', '1']:
                    team = teams[side]['team']
                    team_data = team[0]   # long nested metadata list
                    scoring = team[1]     # dict with points and projections

                    # --- Extract basic info ---
                    team_key = team_data[0]['team_key']
                    team_id = team_data[1]['team_id']
                    team_name = team_data[2]['name']
                    points = float(scoring['team_points']['total'])
                    proj_points = float(scoring['team_projected_points']['total'])

                    # --- Winner flag ---
                    winner = team_key == winner_key

                    # --- Extract manager info ---
                    manager_info = None
                    for item in team_data:
                        if isinstance(item, dict) and 'managers' in item:
                            manager_info = item['managers'][0]['manager']
                            break

                    manager_id = manager_info.get('manager_id') if manager_info else None
                    nickname = manager_info.get('nickname') if manager_info else None
                    guid = manager_info.get('guid') if manager_info else None

                    # First run of a season, collect data to update manager table
                    if previousWeek == 1:
                        num_moves, num_trades = None, None
                        for item in team_data:
                            if isinstance(item, dict):
                                if 'number_of_moves' in item:
                                    num_moves = int(item['number_of_moves'])
                                elif 'number_of_trades' in item:
                                    try:
                                        num_trades = int(item['number_of_trades'])
                                    except ValueError:
                                        num_trades = None
                        managers.append({
                            "manager_guid": guid,
                            "season": currentSeason,
                            "league_id": league_id,
                            "manager_nickname": nickname,
                            "team_id": team_id,
                            "team_key": team_key,
                            "number_of_moves": num_moves,
                            "number_of_trades": num_trades
                        })

                    # --- Add a flattened record ---
                    rows.append({
                        "season": currentSeason,
                        "week": week,
                        "week_start": week_start,
                        "week_end": week_end,
                        "is_playoff": isPlayoff,
                        "matchup_id": key,
                        "team_id": team_id,
                        "team_name": team_name,
                        "team_key": team_key,
                        "manager_id": manager_id,
                        "manager_nickname": nickname,
                        "manager_guid": guid,
                        "points": points,
                        "projected_points": proj_points,
                        "winner": winner,
                        "recap_url": recap_url
                    })

    lastWeekMatchup = pd.DataFrame(rows)
    lastWeekMatchup['game_quality'] = pd.cut(lastWeekMatchup['points'], bins =[0, 120, 150, float('inf')], labels = ['poor', 'average', 'strong'])

In [151]:
if previousWeek == 1:
    manager_seasons_df = pd.DataFrame(managers)
    managers_df = manager_seasons_df[['manager_guid', 'manager_nickname']].drop_duplicates()
    manager_seasons_df = manager_seasons_df.drop(columns=['manager_nickname'])

    # Existing managers
    manager_current = pd.read_sql("SELECT * FROM manager_info.managers", engine)

    # Get any new managers
    managers_new = managers_df[
        ~managers_df['manager_guid'].isin(manager_current['manager_guid'])
    ]
    start = time.time()
    managers_new.to_sql("managers", engine, 'manager_info', if_exists="append", index=False)
    manager_seasons_df.to_sql("manager_seasons", engine, 'manager_info', if_exists="append", index=False)
    end = time.time()
    print(f"New managers and manager seasons insertion completed in {end - start:.2f} seconds")

In [161]:
if previousWeek != 0:
    start = time.time()
    
    lastWeekMatchup.to_sql('matchups', engine, 'manager_info', if_exists='append', index=False, dtype={
        "season": Integer(),            # Season year
        "week": Integer(),              # week number (1–18)
        "week_start": Date(),           # start date of the week
        "week_end": Date(),             # end date of the week
        "is_playoff": Integer(),        # True if this week is in playoffs
        "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
        "team_id": Integer(),           # numeric team ID
        "team_name": String(100),       # e.g., "Hail Mary john"
        "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
        "manager_id": Integer(),        # numeric manager ID (if available)
        "manager_nickname": String(50), # e.g., "Matthew"
        "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
        "points": Float(),              # actual fantasy points scored
        "projected_points": Float(),    # projected points for the matchup
        "winner": String(10),           # 'win', 'loss', 'tie', or None
        "recap_url": String(255),       # URL to Yahoo recap (can be long)
        "game_quality": String(10)      # Designation of strong, average, or poor game
    })

    end = time.time()
    print(f"Insertion completed in {end - start:.2f} seconds")

Insertion completed in 0.06 seconds


In [153]:
if previousWeek != 0:
    previousWeekPlayers = []
    teams = lg.teams()

    for team_name, team in teams.items():
        roster = lg.to_team(team_name).roster(week=previousWeek)
        playerIds = [player["player_id"] for player in roster]
        stats = lg.player_stats(playerIds, req_type='week', week=previousWeek)            
        for player, stat in zip(roster, stats):
            previousWeekPlayers.append({
                        "season": currentSeason,
                        "week": previousWeek,
                        "team_key": team_name,
                        "team_name": team['name'],
                        "manager_guid": team['managers'][0]['manager']['guid'],
                        "manager_name": team['managers'][0]['manager']['nickname'],
                        "player_id": player["player_id"],
                        "player_name": player["name"],
                        "played_position": player['selected_position'],
                        "position_type": stat['position_type'],
                        "pass_yds": stat['Pass Yds'] if 'Pass Yds' in stat else 0,
                        "pass_td": stat['Pass TD'] if 'Pass TD' in stat else 0,
                        "pass_int": stat['Int'] if 'Int' in stat and player['position_type'] == 'O' else 0,
                        "rush_att": stat['Rush Att'] if 'Rush Att' in stat else 0,
                        "rush_yds": stat['Rush Yds'] if 'Rush Yds' in stat else 0,
                        "rush_td": stat['Rush TD'] if 'Rush TD' in stat else 0,
                        "targets": stat['Targets'] if 'Targets' in stat else 0,
                        "rec": stat['Rec'] if 'Rec' in stat else 0,
                        "rec_yds": stat['Rec Yds'] if 'Rec Yds' in stat else 0,
                        "rec_td": stat['Rec TD'] if 'Rec TD' in stat else 0,
                        "offensive_ret_td": stat['Ret TD'] if 'Ret TD' in stat and player['position_type'] == 'O' else 0,
                        "two_pt_conversion": stat['2-PT'] if '2-PT' in stat else 0,
                        "fum_lost": stat['Fum Lost'] if 'Fum Lost' in stat else 0,
                        "fum_ret_td": stat['Fum Ret TD'] if 'Fum Ret TD' in stat else 0,
                        'fg_0_19': stat['FG 0-19'] if 'FG 0-19' in stat else 0,
                        'fg_20_29': stat['FG 20-29'] if 'FG 20-29' in stat else 0,
                        'fg_30_39': stat['FG 30-39'] if 'FG 30-39' in stat else 0,
                        'fg_40_49': stat['FG 40-49'] if 'FG 40-49' in stat else 0,
                        'fg_50_plus': stat['FG 50+'] if 'FG 50+' in stat else 0,
                        'pat_made': stat['PAT Made'] if 'PAT Made' in stat else 0,
                        'pts_allow': stat['Pts Allow'] if 'Pts Allow' in stat else 0,
                        'sack': stat['Sack'] if 'Sack' in stat else 0,
                        'dt_int': stat['Int'] if 'Int' in stat and player['position_type'] == 'DT' else 0,
                        'fum_rec': stat['Fum Rec'] if 'Fum Rec' in stat else 0,
                        'dt_td': stat['TD'] if 'TD' in stat and player['position_type'] == 'DT' else 0,
                        'safe': stat['Safe'] if 'Safe' in stat else 0,
                        'blk_kick': stat['Blk Kick'] if 'Blk Kick' in stat else 0,
                        'dt_ret_td': stat['Ret TD'] if 'Ret TD' in stat and player['position_type'] == 'DT' else 0,
                        'pts_allow_0': stat['Pts Allow 0'] if 'Pts Allow 0' in stat else 0,
                        'pts_allow_1_6': stat['Pts Allow 1-6'] if 'Pts Allow 1-6' in stat else 0,
                        'pts_allow_7_13': stat['Pts Allow 7-13'] if 'Pts Allow 7-13' in stat else 0,
                        'pts_allow_14_20': stat['Pts Allow 14-20'] if 'Pts Allow 14-20' in stat else 0,
                        'pts_allow_21_27': stat['Pts Allow 21-27'] if 'Pts Allow 21-27' in stat else 0,
                        'pts_allow_28_34': stat['Pts Allow 28-34'] if 'Pts Allow 28-34' in stat else 0,
                        'pts_allow_35_plus': stat['Pts Allow 35+'] if 'Pts Allow 35+' in stat else 0,
                        'xpr': stat['XPR'] if 'XPR' in stat else 0,
                        "points": stat['total_points'] if 'total_points' in stat else 0,
                    })

    previousWeekRoster = pd.DataFrame(previousWeekPlayers)

In [156]:
start = time.time()
previousWeekRoster.to_sql('detailed_roster', engine, 'player_info', if_exists='append', index=False, chunksize=200, dtype={
        "season": Integer(),
        "week": Integer(),
        "team_key": String(100),
        "team_name": String(100),
        "manager_guid": String(50),
        "manager_name": String(50),
        "player_id": Integer(),
        "player_name": String(100),
        "played_position": String(10),
        "position_type": String(5),

        # Passing
        "pass_yds": Integer(),
        "pass_td": Integer(),
        "pass_int": Integer(),

        # Rushing
        "rush_att": Integer(),
        "rush_yds": Integer(),
        "rush_td": Integer(),

        # Receiving
        "targets": Integer(),
        "rec": Integer(),
        "rec_yds": Integer(),
        "rec_td": Integer(),

        # Misc offensive
        "offensive_ret_td": Integer(),
        "two_pt_conversion": Integer(),
        "fum_lost": Integer(),
        "fum_ret_td": Integer(),

        # Kicker
        "fg_0_19": Integer(),
        "fg_20_29": Integer(),
        "fg_30_39": Integer(),
        "fg_40_49": Integer(),
        "fg_50_plus": Integer(),
        "pat_made": Integer(),
        "xpr": Integer(),

        # Defense
        "pts_allow": Integer(),
        "sack": Integer(),
        "dt_int": Integer(),
        "fum_rec": Integer(),
        "dt_td": Integer(),
        "safe": Integer(),
        "blk_kick": Integer(),
        "dt_ret_td": Integer(),

        # Defense point tiers
        "pts_allow_0": Integer(),
        "pts_allow_1_6": Integer(),
        "pts_allow_7_13": Integer(),
        "pts_allow_14_20": Integer(),
        "pts_allow_21_27": Integer(),
        "pts_allow_28_34": Integer(),
        "pts_allow_35_plus": Integer(),

        # Final fantasy points
        "points": Float()
    }
)
end = time.time()
print(f"Insertion completed in {end - start:.2f} seconds")

Insertion completed in 0.15 seconds


## Reports

In [157]:
sql = text('''
Select S.*, 
HS.Season as Season_S, HS.Week as Week_S, HS.winningmanager as WinningManager_S, HS.winningpoints as WinningPoints_S,   
HS.losingmanager as LosingManager_S, HS.Losingpoints as LosingPoints_S,
HL.Season as Season_L, HL.Week as Week_L, HL.winningmanager as WinningManager_L, HL.winningpoints as WinningPoints_L,   
HL.losingmanager as LosingManager_L, HL.Losingpoints as LosingPoints_L
From (
Select M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0) as LifetimeWins, COALESCE(R.Losses, RFallBack.Wins, 0) as LifetimeLosses, COALESCE(Min(H.pointdiff), 0) as SmallestPointDifferential, COALESCE(Max(H.pointdiff), 0) as LargestPointDifferential
from vw_currentmatchup M
Left Join lifetimerecord R On M.projectedwinningmanager = R.winningmanager and M.projectedlosingmanager = R.losingmanager
Left Join lifetimerecord RFallBack On M.projectedwinningmanager = RFallBack.losingmanager and M.projectedlosingmanager = RFallBack.winningmanager 
Left Join matchuphistory H On (M.projectedwinningmanager = H.winningmanager or M.projectedwinningmanager = H.losingmanager) and (M.projectedlosingmanager = H.losingmanager or M.projectedlosingmanager = H.winningmanager)
Group By M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0), COALESCE(R.Losses, RFallBack.Wins, 0)
) S
Left Join matchuphistory HL On S.LargestPointDifferential = HL.pointdiff and (S.projectedwinningmanager = HL.winningmanager or S.projectedwinningmanager = HL.losingmanager) and (S.projectedlosingmanager = HL.losingmanager or s.projectedlosingmanager = HL.winningmanager)
Left Join matchuphistory HS On S.SmallestPointDifferential = HS.pointdiff and (S.projectedwinningmanager = HS.winningmanager or S.projectedwinningmanager = HS.losingmanager) and (S.projectedlosingmanager = HS.losingmanager or s.projectedlosingmanager = HS.winningmanager)
''')

report = pd.read_sql(sql, con=engine.connect())

In [158]:
for index, row in report.iterrows():
    if index == 0: print(f"Week {row['week']} of the {row['season']} season")
    print(f"""
    - {row['projectedwinningmanager']} vs {row['projectedlosingmanager']} 
        {f"- Lifetime record ({row['lifetimewins']}-{row['lifetimelosses']})" if not pd.isna(row['season_s']) else '-First matchup'}
        {f"- Smallest point differential {row['smallestpointdifferential']} ({row['winningmanager_s']} beat {row['losingmanager_s']} {row['winningpoints_s']} to {row['losingpoints_s']} in week {int(row['week_s'])} of the {row['season_s']} season)" if not pd.isna(row['season_s']) else ''}
        {f"- Largest point differential {row['largestpointdifferential']} ({row['winningmanager_l']} beat {row['losingmanager_l']} {row['winningpoints_l']} to {row['losingpoints_l']} in week {int(row['week_l'])} of the {row['season_l']} season)" if not pd.isna(row['season_s']) else ''}""")

Week 15 of the 2025 season

    - Nick vs Liam 
        - Lifetime record (0-1)
        - Smallest point differential 23.04 (Liam beat Nick 143.16 to 120.12 in week 8 of the 2025 season)
        - Largest point differential 23.04 (Liam beat Nick 143.16 to 120.12 in week 8 of the 2025 season)

    - Billy vs Duffy 
        - Lifetime record (2-4)
        - Smallest point differential 9.76 (Billy beat Duffy 124.28 to 114.52 in week 16 of the 2024 season)
        - Largest point differential 34.92 (Duffy beat Billy 162.66 to 127.74 in week 11 of the 2024 season)


## NoSQL Replication - Part 5.1

### Database

In [5]:
from pymongo import MongoClient

In [6]:
# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["MercyFantasy"]
managers = db["managers"]
manager_seasons = db["manager_seasons"]
matchups = db["matchups"]
detailed_roster = db["detailed_roster"]

noSQLDict = {
    'managers': managers,
    'manager_seasons': manager_seasons,
    'matchups': matchups,
    'detailed_roster': detailed_roster
}

In [ ]:
# Data transfer
## Get tables
tables = list(noSQLDict.keys())

for table in tables:
    # Get data
    schema = 'manager_info' if table != 'detailed_roster' else 'player_info'
    df = pd.read_sql(text(f'Select * From {schema}.{table}'), engine.connect(), parse_dates=['week_start', 'week_end'])

    # Handle dates for MongoDB
    date_columns = df.select_dtypes(include=['datetime64']).columns
    for col in date_columns:
        df[col] = pd.to_datetime(df[col])

    # Convert to dictionary and insert into MongoDB
    records = df.to_dict('records')
    noSQLDict[table].insert_many(records)

### Simple Table Search

In [36]:
# Create index to match PostgreSQL one on primary keys
detailed_roster.create_index({
  'season': 1,
  'week': 1,
  'player_id': 1
})

'season_1_week_1_player_id_1'

In [92]:
playerweek = detailed_roster.find({'season': 2023, 'week':17, 'player_id': 40063})
playerweek.explain()

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'MercyFantasy.detailed_roster',
  'parsedQuery': {'$and': [{'player_id': {'$eq': 40063}},
    {'season': {'$eq': 2023}},
    {'week': {'$eq': 17}}]},
  'indexFilterSet': False,
  'queryHash': 'FA2651C7',
  'planCacheShapeHash': 'FA2651C7',
  'planCacheKey': '1EAA3012',
  'optimizationTimeMillis': 3,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'season': 1, 'week': 1, 'player_id': 1},
    'indexName': 'season_1_week_1_player_id_1',
    'isMultiKey': False,
    'multiKeyPaths': {'season': [], 'week': [], 'player_id': []},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'season': ['[2023, 2023]'],
     'week': ['[17, 17]'],
 

In [94]:
somePgPLayer =  text("""
EXPLAIN ANALYZE
SELECT *
FROM player_info.detailed_roster
WHERE season = 2023
and week = 17
and player_id = 40063
""")

somePgPLayer_df = pd.read_sql(somePgPLayer, engine.connect())

for i, row in somePgPLayer_df.iterrows():
    print(row['QUERY PLAN'])

Index Scan using detailed_roster_pkey on detailed_roster  (cost=0.29..18.32 rows=1 width=256) (actual time=0.109..0.120 rows=1 loops=1)
  Index Cond: ((season = 2023) AND (week = 17) AND (player_id = 40063))
Planning Time: 3.387 ms
Execution Time: 0.232 ms


Here is a super basic query for both DBs, querying for and retrieving one record. Both queries cleanly utilize indexes (PostgreSQL uses index on the primary key named `detailed_roster_pkey` and MongoDB uses and IXSCAN on `season_1_week_1_player_id_1`) on the search parameters, so execution time was extremly low (4 ms for MongoDB and 0.232 for PostgreSQL). The very small difference may be due to MongoDB materializing a BSON object to run the query or simple issues measuring sub second time. Basically, these queries are highly efficient! The optimization time for PostgreSQL was 3.387 ms and around 3 ms for MongoDB (`optimizationTimeMillis`), so very similar amount of time to optimize the query before running. This example shows how highly optimized both databases are. With the same indexes available, both databases are taking the same steps to retrieve the record as fast as possible.

### Join Query

In [102]:
pipeline = [
    {
        "$lookup": {
            "from": "detailed_roster",
            "let": {
                "team_key": "$team_key",
                "week": "$week",
                "season": "$season"
            },
            "pipeline": [
                {
                    "$match": {
                        "$expr": {
                            "$and": [
                                { "$eq": ["$team_key", "$$team_key"] },
                                { "$eq": ["$week", "$$week"] },
                                { "$eq": ["$season", "$$season"] }
                            ]
                        }
                    }
                }
            ],
            "as": "roster"
        }
    },
    {
        "$unwind": {
            "path": "$roster",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$group": {
            "_id": {
                "season": "$season",
                "manager_nickname": "$manager_nickname",
                "player_name": "$roster.player_name"
            },
            "Points": { "$sum": "$roster.points" }
        }
    },
    {
        "$project": {
            "_id": 0,
            "Season": "$_id.season",
            "Manager_nickname": "$_id.manager_nickname",
            "Player_Name": "$_id.player_name",
            "Points": { "$round": ["$Points", 2] }
        }
    }
]

results = matchups.aggregate(pipeline)

for r in list(results)[0:5]:
    print(r)

{'Season': 2023, 'Manager_nickname': 'daniel', 'Player_Name': 'Philadelphia', 'Points': 66.0}
{'Season': 2023, 'Manager_nickname': 'john', 'Player_Name': 'Amari Cooper', 'Points': 156.2}
{'Season': 2024, 'Manager_nickname': 'Joseph', 'Player_Name': 'Pat Freiermuth', 'Points': 26.8}
{'Season': 2023, 'Manager_nickname': 'Billy', 'Player_Name': 'Odell Beckham', 'Points': 27.3}
{'Season': 2023, 'Manager_nickname': 'chris', 'Player_Name': 'Zach Charbonnet', 'Points': 32.4}


In [99]:
explain_output = db.command({
    "explain": {
        "aggregate": "matchups",
        "pipeline": pipeline,
        "cursor": {}
    },
    "verbosity": "executionStats"
})
explain_output

{'explainVersion': '1',
 'stages': [{'$cursor': {'queryPlanner': {'namespace': 'MercyFantasy.matchups',
     'parsedQuery': {},
     'indexFilterSet': False,
     'queryHash': '2FAB5EB5',
     'planCacheShapeHash': '2FAB5EB5',
     'planCacheKey': 'BA131F4F',
     'optimizationTimeMillis': 0,
     'maxIndexedOrSolutionsReached': False,
     'maxIndexedAndSolutionsReached': False,
     'maxScansToExplodeReached': False,
     'prunedSimilarIndexes': False,
     'winningPlan': {'isCached': False,
      'stage': 'PROJECTION_SIMPLE',
      'transformBy': {'manager_nickname': 1,
       'season': 1,
       'team_key': 1,
       'week': 1,
       '_id': 0},
      'inputStage': {'stage': 'COLLSCAN', 'direction': 'forward'}},
     'rejectedPlans': []},
    'executionStats': {'executionSuccess': True,
     'nReturned': 454,
     'executionTimeMillis': 443,
     'totalKeysExamined': 0,
     'totalDocsExamined': 454,
     'executionStages': {'isCached': False,
      'stage': 'PROJECTION_SIMPLE',
  

In [104]:
joinSQL = text('''
EXPLAIN ANALYZE
Select m.Season, m.Manager_nickname, r.Player_Name, Round(Sum(r.Points)::Numeric, 2) as Points
From manager_info.matchups m
Left Join player_info.detailed_roster r On m.team_key = r.team_key and m.week = r.week and m.season = r.season
Group By m.Season, m.Manager_nickname, r.Player_Name
''')

joinEx = pd.read_sql(joinSQL, engine.connect())

for i, row in joinEx.iterrows():
    print(row['QUERY PLAN'])

HashAggregate  (cost=572.68..610.39 rows=2514 width=55) (actual time=51.043..52.462 rows=1380 loops=1)
  Group Key: m.season, m.manager_nickname, r.player_name
  Batches: 1  Memory Usage: 369kB
  ->  Nested Loop Left Join  (cost=0.30..547.54 rows=2514 width=31) (actual time=0.461..46.017 rows=7549 loops=1)
        ->  Seq Scan on matchups m  (cost=0.00..32.54 rows=454 width=31) (actual time=0.218..0.557 rows=454 loops=1)
        ->  Memoize  (cost=0.30..10.36 rows=6 width=46) (actual time=0.045..0.095 rows=17 loops=454)
              Cache Key: m.team_key, m.week, m.season
              Cache Mode: logical
              Hits: 0  Misses: 454  Evictions: 0  Overflows: 0  Memory Usage: 665kB
              ->  Index Scan using detailed_roster_pkey on detailed_roster r  (cost=0.29..10.35 rows=6 width=46) (actual time=0.043..0.083 rows=17 loops=454)
                    Index Cond: ((season = m.season) AND (week = m.week))
                    Filter: ((m.team_key)::text = (team_key)::text)
  

This example is a bit more complex since there are joins and aggregations involved. PostgreSQL uses nested loops and a technique called memoization (optimization technique which stores the results of expensive function calls and returns the cached result when the same inputs occur again) to avoid repeated index scans, which saves time when joining. MongoDB’s total execution time of 439 ms is significantly higher than PostgreSQL’s 54.639 ms, but the difference is not caused by MongoDB examining more rows. Both systems examine roughly 75,000 candidate rows, but PostgreSQL applies a more efficient execution strategy via its use of the Memoize operator and an index whose leading columns align with the join conditions. Both systems have low planning times as well (3.187 ms for PostgresSQL and about 0 for MongoDB). PostgreSQL performs an index range scan on (season, week) and then filters on team_key in memory, which reduces each lookup to approximately 17 matching rows after filtering, even though about 167 rows are inspected per lookup. MongoDB on the other hand performs a lookup for each outer document and must scan many more candidate documents in the joined collection due to the missing index, resulting in substantially more work in query. Both queries aggregate to the season-team level, and return 1,380 rows. It seems clear that join heavy queries are better in traditional SQL compared to NoSQL, due to memoization and the ability to use indexes effectively.

### NoSQL Document versus SQL

Create one true non-flat document model for MongoDB!

In [ ]:
# Managers document

manager_rosters = db["manager_rosters"]

jsonQuery = text("""
SELECT 
    json_build_object(
        'manager_name', m.propername,
        'seasons', json_agg(
            json_build_object(
                'season', s.season,
                'weeks',
                (
                    SELECT json_agg(
                        json_build_object(
                            'week', w.week,
                            'players',
                            (
                                SELECT json_agg(
                                    json_build_object(
                                        'player_id', dr3.player_id,
                                        'player_name', dr3.player_name,
                                        'played_position', dr3.played_position,
                                        'points', dr3.points
                                    )
                                    ORDER BY dr3.player_id
                                )
                                FROM player_info.detailed_roster dr3
                                WHERE dr3.manager_guid = s.manager_guid
                                  AND dr3.season = s.season
                                  AND dr3.week = w.week
                            )
                        )
                        ORDER BY w.week
                    )
                    FROM (
                        SELECT DISTINCT week
                        FROM player_info.detailed_roster dr2
                        WHERE dr2.manager_guid = s.manager_guid
                          AND dr2.season = s.season
                    ) w
                )
            )
            ORDER BY s.season
        )
    ) AS doc
FROM (
    SELECT DISTINCT manager_guid, season
    FROM player_info.detailed_roster
) s
JOIN manager_map m ON m.manager_guid = s.manager_guid
GROUP BY m.propername, s.manager_guid
ORDER BY m.propername
"""
)

manager_roster_df = pd.read_sql(jsonQuery, engine.connect())

docs = manager_roster_df["doc"].tolist()
manager_rosters.insert_many(docs)

InsertManyResult([ObjectId('6938fbb8f3594c66aa182541'), ObjectId('6938fbb8f3594c66aa182542'), ObjectId('6938fbb8f3594c66aa182543'), ObjectId('6938fbb8f3594c66aa182544'), ObjectId('6938fbb8f3594c66aa182545'), ObjectId('6938fbb8f3594c66aa182546'), ObjectId('6938fbb8f3594c66aa182547'), ObjectId('6938fbb8f3594c66aa182548'), ObjectId('6938fbb8f3594c66aa182549'), ObjectId('6938fbb8f3594c66aa18254a'), ObjectId('6938fbb8f3594c66aa18254b'), ObjectId('6938fbb8f3594c66aa18254c')], acknowledged=True)

In [111]:
# Create an index to quickly look for managers and seasons!
## PostgreSQL has one as well on manager guid
manager_rosters.create_index({"manager_name": 1})

'manager_name_1'

In [137]:
# Query and explain
noSQLsomemanager = manager_rosters.find({"manager_name": "Duffy"})
noSQLsomemanager.explain()

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'MercyFantasy.manager_rosters',
  'parsedQuery': {'manager_name': {'$eq': 'Duffy'}},
  'indexFilterSet': False,
  'queryHash': 'B7765225',
  'planCacheShapeHash': 'B7765225',
  'planCacheKey': 'C9D9AE1A',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'manager_name': 1},
    'indexName': 'manager_name_1',
    'isMultiKey': False,
    'multiKeyPaths': {'manager_name': []},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'manager_name': ['["Duffy", "Duffy"]']}}},
  'rejectedPlans': []},
 'executionStats': {'executionSuccess': True,
  'nReturned': 1,
  'executionTimeMillis': 0,
  'totalKeysExamine

In [139]:
postgresomeManager =  text("""
EXPLAIN ANALYZE
SELECT *
FROM player_info.detailed_roster r
WHERE r.manager_guid = 'GOBJY4A3RDGQYERM2ETTMRC7YY'
""")

someManager_df = pd.read_sql(postgresomeManager, engine.connect())

for i, row in someManager_df.iterrows():
    print(row['QUERY PLAN'])

Bitmap Heap Scan on detailed_roster r  (cost=19.06..675.46 rows=874 width=256) (actual time=0.969..1.837 rows=874 loops=1)
  Recheck Cond: ((manager_guid)::text = 'GOBJY4A3RDGQYERM2ETTMRC7YY'::text)
  Heap Blocks: exact=86
  ->  Bitmap Index Scan on idx_manager_detailed  (cost=0.00..18.84 rows=874 width=0) (actual time=0.868..0.868 rows=874 loops=1)
        Index Cond: ((manager_guid)::text = 'GOBJY4A3RDGQYERM2ETTMRC7YY'::text)
Planning Time: 2.386 ms
Execution Time: 2.071 ms


For the final query I pulled all player a particular manager (`Duffy` or guid `GOBJY4A3RDGQYERM2ETTMRC7YY`) has had on their roster. In PostgreSQL I queried the detailed_roster table which has one player per week. In MongoDB, I simply queried the document from the JSON constructed collection manager_rosters, which contains one document per manager with the complete historical roster embedded directly inside the document. Both queries are able to leverage indexes on the search field for optimized execution, but MongoDB only needed to locate a single indexed document, so returned it in nearly 0 ms (optimization planning also took about 0 ms). That illustrates the strengths of the NoSQL document model: storing related, semi-structured data in a single document using key–value structures can make certain retrieval patterns extremely fast, especially in bid data and real time analytics where large amounts of related information can be with one key lookup. Postgres took around 2.1 ms (optimization planning took about 2.3 ms); it utilized a bitmap index scan on manager_guid and examined/returned 874 rows in the process. While this is highly efficient, PostgreSQL still needs to retrieve every entry while MongoDB only needs to find and return one document. These time gap would be expounded as well as table sizes grow in PostgreSQL!